In [ ]:
# ============================================================

# Strict Class 1 shadow disease-pair extraction

# Paper-ready version

#

# Objective:

# Find pairs of rare diseases sharing the same

# biochemical reaction profile but differing in HPO annotation availability:

#

# target disease: n_HPO = 0

# donor disease:  n_HPO > 0

#

# Strict matching criteria:

# Same Gene

# Same UniProt Entry

# Same EC

# Same Rhea_ID

# Same Cofactor

# Same Pathway

#

# Main output:

# class1_single_profile_shadow_pairs_paper_ready.csv

# ============================================================

import pandas as pd

import numpy as np

import re

from pathlib import Path

# ============================================================

# INPUT / OUTPUT

# ============================================================

INPUT_FILE = Path(

    "data/dataset_finale_Stoichiometry_CLEAN.csv"

)

OUTPUT_DIR = INPUT_FILE.parent / "reaction_profile_similarity_results_paper_ready"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================

# CLEANING FUNCTIONS

# ============================================================

MISSING_VALUES = {"", "na", "nan", "none", "null", "n/a", "-", ".", "[]"}

def is_missing(x):

    if pd.isna(x):

        return True

    return str(x).strip().lower() in MISSING_VALUES

def clean_text(x):

    if is_missing(x):

        return ""

    x = str(x).strip()

    x = re.sub(r"\s+", " ", x)

    return x

def normalize_id(x):

    return clean_text(x).upper()

def normalize_text(x):

    return clean_text(x)

def normalize_ec(x):

    """

    Normalize EC for strict matching without removing the dots.

    Example:

    ' 2.6.1.52 ' -> '2.6.1.52'

    Do not turn 2.6.1.52 into 26152, since removing dots

    may cause collisions between different EC numbers.

    """

    x = clean_text(x)

    if not x:

        return ""

    parts = re.split(r";|\||,", x)

    cleaned = []

    for p in parts:

        p = clean_text(p)

        if not p:

            continue

        p = p.replace(" ", "")

        p = re.sub(r"\.+", ".", p)

        p = re.sub(r"\.$", "", p)

        cleaned.append(p)

    return ";".join(sorted(set(cleaned)))

def normalize_multivalue(x, lowercase=True):

    """

    Normalize multivalue fields such as Cofactor and Pathway.

    'A;B' and 'B;A' become equivalent.

    """

    x = clean_text(x)

    if not x:

        return ""

    x = x.replace(" .", ".")

    x = re.sub(r"\.+$", "", x)

    x = re.sub(r"\s+", " ", x)

    parts = re.split(r";|\|", x)

    cleaned_parts = []

    for p in parts:

        p = clean_text(p)

        if not p:

            continue

        p = p.replace(" .", ".")

        p = re.sub(r"\.+$", "", p)

        p = re.sub(r"\s+", " ", p).strip()

        if lowercase:

            p = p.lower()

        if p:

            cleaned_parts.append(p)

    return ";".join(sorted(set(cleaned_parts)))

def unique_join(values, sep=";"):

    vals = []

    for v in values:

        v = clean_text(v)

        if v:

            vals.append(v)

    return sep.join(sorted(set(vals)))

# ============================================================

# LOAD DATASET

# ============================================================

df = pd.read_csv(

    INPUT_FILE,

    sep=";",

    dtype=str,

    keep_default_na=False,

    low_memory=False,

    encoding="utf-8-sig"

)

df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]

# Harmonize possible column names

rename_map = {

    "ChEBI substrates": "ChEBI_substrates",

    "ChEBI product": "ChEBI_product",

    "ChEBI products": "ChEBI_products",

    "ChEBI cofactor": "ChEBI_cofactor",

    "product_nams": "Product_names",

    "product_names": "Product_names"

}

df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = [

    "ORPHAcode",

    "DiseaseName",

    "Gene",

    "Entry",

    "Rhea_ID",

    "Reaction",

    "EC",

    "Cofactor",

    "Cofactor_type",

    "Pathway",

    "HPO_ID",

    "HPO_Label"

]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:

    raise ValueError(f"Missing columns in input dataset: {missing_cols}")

# ============================================================

# NORMALIZATION

# ============================================================

df["ORPHAcode_norm"] = df["ORPHAcode"].apply(normalize_id)

df["DiseaseName_norm"] = df["DiseaseName"].apply(normalize_text)

df["Gene_norm"] = df["Gene"].apply(normalize_id)

df["Entry_norm"] = df["Entry"].apply(normalize_id)

df["Rhea_ID_norm"] = df["Rhea_ID"].apply(normalize_id)

df["Reaction_norm"] = df["Reaction"].apply(normalize_text)

df["EC_display"] = df["EC"].apply(normalize_text)

df["EC_norm"] = df["EC"].apply(normalize_ec)

df["Cofactor_norm"] = df["Cofactor"].apply(lambda x: normalize_multivalue(x, lowercase=True))

df["Cofactor_type_norm"] = df["Cofactor_type"].apply(lambda x: normalize_multivalue(x, lowercase=True))

df["Pathway_norm"] = df["Pathway"].apply(lambda x: normalize_multivalue(x, lowercase=True))

df["HPO_ID_norm"] = df["HPO_ID"].apply(normalize_id)

df["HPO_Label_norm"] = df["HPO_Label"].apply(normalize_text)

df["has_HPO"] = ~df["HPO_ID_norm"].apply(is_missing)

df["has_EC"] = ~df["EC_norm"].apply(is_missing)

df["has_Pathway"] = ~df["Pathway_norm"].apply(is_missing)

df["has_Cofactor"] = ~df["Cofactor_norm"].apply(is_missing)

# ============================================================

# BASIC CHECKS

# ============================================================

print("\n===== BASIC DATASET CHECK =====")

print("Raw rows:", len(df))

print("Unique diseases:", df["ORPHAcode_norm"].nunique())

print("Unique genes:", df["Gene_norm"].nunique())

print("Unique UniProt entries:", df["Entry_norm"].nunique())

print("Unique Rhea reactions:", df["Rhea_ID_norm"].nunique())

print("\n===== MISSINGNESS =====")

print("Rows with HPO:", int(df["has_HPO"].sum()))

print("Rows without HPO:", int((~df["has_HPO"]).sum()))

print("Rows with EC:", int(df["has_EC"].sum()))

print("Rows without EC:", int((~df["has_EC"]).sum()))

print("Rows with pathway:", int(df["has_Pathway"].sum()))

print("Rows without pathway:", int((~df["has_Pathway"]).sum()))

print("Rows with cofactor:", int(df["has_Cofactor"].sum()))

print("Rows without cofactor:", int((~df["has_Cofactor"]).sum()))

# ============================================================

# HPO COUNT PER DISEASE

# ============================================================

hpo_per_disease = (

    df[df["has_HPO"]]

    .groupby("ORPHAcode_norm", dropna=False)

    .agg(

        n_HPO=("HPO_ID_norm", "nunique"),

        HPO_IDs=("HPO_ID_norm", lambda x: unique_join(x, sep=";")),

        HPO_Labels=("HPO_Label_norm", lambda x: unique_join(x, sep=";"))

    )

    .reset_index()

)

# One disease name per ORPHAcode

all_diseases = (

    df.groupby("ORPHAcode_norm", dropna=False)

    .agg(

        DiseaseName_norm=("DiseaseName_norm", lambda x: unique_join(x, sep=" || "))

    )

    .reset_index()

)

disease_hpo = all_diseases.merge(

    hpo_per_disease,

    on="ORPHAcode_norm",

    how="left"

)

disease_hpo["n_HPO"] = disease_hpo["n_HPO"].fillna(0).astype(int)

disease_hpo["HPO_IDs"] = disease_hpo["HPO_IDs"].fillna("")

disease_hpo["HPO_Labels"] = disease_hpo["HPO_Labels"].fillna("")

disease_hpo["HPO_status"] = np.where(

    disease_hpo["n_HPO"] > 0,

    "phenotype_rich",

    "phenotype_poor"

)

print("\n===== DISEASE HPO STATUS =====")

print(disease_hpo["HPO_status"].value_counts())

# ============================================================

# DISEASE-REACTION PROFILES

# ============================================================

profile_cols = [

    "ORPHAcode_norm",

    "Gene_norm",

    "Entry_norm",

    "EC_norm",

    "EC_display",

    "Rhea_ID_norm",

    "Reaction_norm",

    "Cofactor_norm",

    "Cofactor_type_norm",

    "Pathway_norm",

    "has_EC",

    "has_Pathway",

    "has_Cofactor"

]

disease_profiles = (

    df[profile_cols]

    .drop_duplicates()

    .merge(

        disease_hpo[

            [

                "ORPHAcode_norm",

                "DiseaseName_norm",

                "n_HPO",

                "HPO_status",

                "HPO_IDs",

                "HPO_Labels"

            ]

        ],

        on="ORPHAcode_norm",

        how="left"

    )

)

disease_profiles.to_csv(

    OUTPUT_DIR / "disease_reaction_profiles_debug.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

print("\nDisease-reaction profiles:", len(disease_profiles))

print("Diseases in profiles:", disease_profiles["ORPHAcode_norm"].nunique())

# ============================================================

# DIAGNOSTIC: SAME GENE + ENTRY + RHEA

# ============================================================

basic_group_cols = [

    "Gene_norm",

    "Entry_norm",

    "Rhea_ID_norm"

]

basic_groups = (

    disease_profiles

    .groupby(basic_group_cols, dropna=False)

    .agg(

        n_diseases=("ORPHAcode_norm", "nunique"),

        n_poor=("n_HPO", lambda x: int((x == 0).sum())),

        n_rich=("n_HPO", lambda x: int((x > 0).sum())),

        diseases=("DiseaseName_norm", lambda x: unique_join(x, sep=" || ")),

        cofactors=("Cofactor_norm", lambda x: unique_join(x, sep=" || ")),

        pathways=("Pathway_norm", lambda x: unique_join(x, sep=" || "))

    )

    .reset_index()

)

basic_shadow_groups = basic_groups[

    (basic_groups["n_diseases"] >= 2) &

    (basic_groups["n_poor"] > 0) &

    (basic_groups["n_rich"] > 0)

].copy()

basic_shadow_groups.to_csv(

    OUTPUT_DIR / "debug_candidate_groups_same_gene_entry_rhea.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

print("\nCandidate groups same Gene+Entry+Rhea:", len(basic_shadow_groups))

# ============================================================

# STRICT SHADOW DISEASE-PAIR EXTRACTION

# Strict profile:

# Gene + Entry + EC + Rhea_ID + Cofactor + Pathway

#

# Cofactor_type is retained for interpretation/output,

# but it is not used as a blocking key here.

# Add "Cofactor_type_norm" to strict_group_cols if you want

# an even stricter matching.

# ============================================================

strict_group_cols = [

    "Gene_norm",

    "Entry_norm",

    "EC_norm",

    "Rhea_ID_norm",

    "Cofactor_norm",

    "Pathway_norm"

]

disease_profiles["strict_profile_key"] = (

    disease_profiles["Gene_norm"] + "|" +

    disease_profiles["Entry_norm"] + "|" +

    disease_profiles["EC_norm"] + "|" +

    disease_profiles["Rhea_ID_norm"] + "|" +

    disease_profiles["Cofactor_norm"] + "|" +

    disease_profiles["Pathway_norm"]

)

all_profile_matches = []

for key, group in disease_profiles.groupby(strict_group_cols, dropna=False):

    # Reject incomplete strict biochemical profiles

    if any(group[col].iloc[0] == "" for col in strict_group_cols):

        continue

    phenotype_poor = (

        group[group["n_HPO"] == 0]

        .drop_duplicates("ORPHAcode_norm")

    )

    phenotype_rich = (

        group[group["n_HPO"] > 0]

        .drop_duplicates("ORPHAcode_norm")

    )

    if phenotype_poor.empty or phenotype_rich.empty:

        continue

    for _, poor in phenotype_poor.iterrows():

        for _, rich in phenotype_rich.iterrows():

            all_profile_matches.append({

                "target_ORPHAcode": poor["ORPHAcode_norm"],

                "target_DiseaseName": poor["DiseaseName_norm"],

                "target_n_HPO": int(poor["n_HPO"]),

                "donor_ORPHAcode": rich["ORPHAcode_norm"],

                "donor_DiseaseName": rich["DiseaseName_norm"],

                "donor_n_HPO": int(rich["n_HPO"]),

                "donor_HPO_IDs": rich["HPO_IDs"],

                "donor_HPO_Labels": rich["HPO_Labels"],

                "Gene": poor["Gene_norm"],

                "Entry": poor["Entry_norm"],

                "EC_norm": poor["EC_norm"],

                "EC": poor["EC_display"],

                "Rhea_ID": poor["Rhea_ID_norm"],

                "Reaction": poor["Reaction_norm"],

                "Cofactor": poor["Cofactor_norm"],

                "Cofactor_type": poor["Cofactor_type_norm"],

                "Pathway": poor["Pathway_norm"],

                "strict_profile_key": poor["strict_profile_key"],

                "sim_RP_strict": 1.0,

                "pair_status": "strict_shadow_profile_match"

            })

expected_match_cols = [

    "target_ORPHAcode",

    "target_DiseaseName",

    "target_n_HPO",

    "donor_ORPHAcode",

    "donor_DiseaseName",

    "donor_n_HPO",

    "donor_HPO_IDs",

    "donor_HPO_Labels",

    "Gene",

    "Entry",

    "EC_norm",

    "EC",

    "Rhea_ID",

    "Reaction",

    "Cofactor",

    "Cofactor_type",

    "Pathway",

    "strict_profile_key",

    "sim_RP_strict",

    "pair_status"

]

all_profile_matches_df = pd.DataFrame(all_profile_matches)

if all_profile_matches_df.empty:

    all_profile_matches_df = pd.DataFrame(columns=expected_match_cols)

all_profile_matches_df.to_csv(

    OUTPUT_DIR / "all_strict_shadow_profile_matches.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

# ============================================================

# COLLAPSE PROFILE-LEVEL MATCHES TO DISEASE-LEVEL PAIRS

# ============================================================

if not all_profile_matches_df.empty:

    pair_summary = (

        all_profile_matches_df

        .groupby(

            [

                "target_ORPHAcode",

                "target_DiseaseName",

                "target_n_HPO",

                "donor_ORPHAcode",

                "donor_DiseaseName",

                "donor_n_HPO"

            ],

            dropna=False

        )

        .agg(

            n_shared_strict_profiles=("strict_profile_key", "nunique"),

            n_shared_genes=("Gene", "nunique"),

            n_shared_entries=("Entry", "nunique"),

            n_shared_ECs=("EC_norm", "nunique"),

            n_shared_reactions=("Rhea_ID", "nunique"),

            shared_genes=("Gene", lambda x: unique_join(x, sep=";")),

            shared_entries=("Entry", lambda x: unique_join(x, sep=";")),

            shared_ECs=("EC_norm", lambda x: unique_join(x, sep=";")),

            shared_rhea=("Rhea_ID", lambda x: unique_join(x, sep=";")),

            shared_cofactors=("Cofactor", lambda x: unique_join(x, sep=";")),

            shared_pathways=("Pathway", lambda x: unique_join(x, sep=" || ")),

            donor_HPO_IDs=("donor_HPO_IDs", "first"),

            donor_HPO_Labels=("donor_HPO_Labels", "first")

        )

        .reset_index()

    )

else:

    pair_summary = pd.DataFrame(columns=[

        "target_ORPHAcode",

        "target_DiseaseName",

        "target_n_HPO",

        "donor_ORPHAcode",

        "donor_DiseaseName",

        "donor_n_HPO",

        "n_shared_strict_profiles",

        "n_shared_genes",

        "n_shared_entries",

        "n_shared_ECs",

        "n_shared_reactions",

        "shared_genes",

        "shared_entries",

        "shared_ECs",

        "shared_rhea",

        "shared_cofactors",

        "shared_pathways",

        "donor_HPO_IDs",

        "donor_HPO_Labels"

    ])

pair_summary["pair_class"] = np.where(

    pair_summary["n_shared_strict_profiles"].fillna(0).astype(int) == 1,

    "Class 1 - single-profile shadow pair",

    "Excluded - multi-profile or complex-associated pair"

)

# ============================================================

# CLASS 1 SINGLE-PROFILE SHADOW PAIRS

# ============================================================

class1_pairs = pair_summary[

    pair_summary["n_shared_strict_profiles"].fillna(0).astype(int) == 1

].copy()

multi_profile_pairs = pair_summary[

    pair_summary["n_shared_strict_profiles"].fillna(0).astype(int) > 1

].copy()

# Add profile information back to Class 1 pairs

if not all_profile_matches_df.empty:

    profile_info = (

        all_profile_matches_df

        .groupby(["target_ORPHAcode", "donor_ORPHAcode"], dropna=False)

        .agg(

            Gene=("Gene", "first"),

            Entry=("Entry", "first"),

            EC_norm=("EC_norm", "first"),

            EC=("EC", lambda x: unique_join(x, sep=";")),

            Rhea_ID=("Rhea_ID", "first"),

            Reaction=("Reaction", lambda x: unique_join(x, sep=" || ")),

            Cofactor=("Cofactor", "first"),

            Cofactor_type=("Cofactor_type", lambda x: unique_join(x, sep=";")),

            Pathway=("Pathway", "first"),

            sim_RP_strict=("sim_RP_strict", "first")

        )

        .reset_index()

    )

else:

    profile_info = pd.DataFrame(columns=[

        "target_ORPHAcode",

        "donor_ORPHAcode",

        "Gene",

        "Entry",

        "EC_norm",

        "EC",

        "Rhea_ID",

        "Reaction",

        "Cofactor",

        "Cofactor_type",

        "Pathway",

        "sim_RP_strict"

    ])

if not class1_pairs.empty:

    class1_final = class1_pairs.merge(

        profile_info,

        on=["target_ORPHAcode", "donor_ORPHAcode"],

        how="left"

    )

else:

    class1_final = pd.DataFrame()

final_cols = [

    "target_ORPHAcode",

    "target_DiseaseName",

    "target_n_HPO",

    "donor_ORPHAcode",

    "donor_DiseaseName",

    "donor_n_HPO",

    "Gene",

    "Entry",

    "EC_norm",

    "EC",

    "Rhea_ID",

    "Reaction",

    "Cofactor",

    "Cofactor_type",

    "Pathway",

    "donor_HPO_IDs",

    "donor_HPO_Labels",

    "n_shared_strict_profiles",

    "n_shared_genes",

    "n_shared_entries",

    "n_shared_ECs",

    "n_shared_reactions",

    "sim_RP_strict",

    "pair_class"

]

if class1_final.empty:

    class1_final = pd.DataFrame(columns=final_cols)

else:

    class1_final = class1_final[final_cols].copy()

# ============================================================

# PROFILE-LEVEL STATISTICS

# ============================================================

profile_stats = (

    disease_profiles

    .groupby(strict_group_cols, dropna=False)

    .agg(

        n_diseases=("ORPHAcode_norm", "nunique"),

        n_phenotype_poor=("n_HPO", lambda x: int((x == 0).sum())),

        n_phenotype_rich=("n_HPO", lambda x: int((x > 0).sum())),

        diseases=("DiseaseName_norm", lambda x: unique_join(x, sep=" || ")),

        ORPHAcodes=("ORPHAcode_norm", lambda x: unique_join(x, sep=" || "))

    )

    .reset_index()

)

profile_stats["is_mixed_HPO_group"] = (

    (profile_stats["n_diseases"] >= 2) &

    (profile_stats["n_phenotype_poor"] > 0) &

    (profile_stats["n_phenotype_rich"] > 0)

)

# ============================================================

# SAVE FILES

# ============================================================

class1_final.to_csv(

    OUTPUT_DIR / "class1_single_profile_shadow_pairs_paper_ready.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

pair_summary.to_csv(

    OUTPUT_DIR / "all_disease_level_shadow_pairs_summary.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

multi_profile_pairs.to_csv(

    OUTPUT_DIR / "excluded_multi_profile_or_complex_shadow_pairs.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

profile_stats.to_csv(

    OUTPUT_DIR / "strict_profile_group_statistics.csv",

    sep=";",

    index=False,

    encoding="utf-8-sig"

)

# ============================================================

# FINAL REPORT

# ============================================================

print("\n=== STRICT SHADOW DISEASE-PAIR EXTRACTION ===")

print(f"Total diseases: {df['ORPHAcode_norm'].nunique()}")

print(f"Diseases with HPO: {(disease_hpo['n_HPO'] > 0).sum()}")

print(f"Diseases without HPO: {(disease_hpo['n_HPO'] == 0).sum()}")

print(f"Disease-reaction profiles: {len(disease_profiles)}")

print(f"\nStrict profile-level matches: {len(all_profile_matches_df)}")

print(f"Unique disease-level shadow pairs: {len(pair_summary)}")

print(f"Class 1 single-profile shadow pairs: {len(class1_final)}")

print(f"Excluded multi-profile / complex-associated pairs: {len(multi_profile_pairs)}")

print("\nStrict profile key used:")

print("Gene + Entry + EC + Rhea_ID + Cofactor + Pathway")

print("\nMain files saved:")

print(OUTPUT_DIR / "all_strict_shadow_profile_matches.csv")

print(OUTPUT_DIR / "class1_single_profile_shadow_pairs_paper_ready.csv")

print(OUTPUT_DIR / "excluded_multi_profile_or_complex_shadow_pairs.csv")

print(OUTPUT_DIR / "all_disease_level_shadow_pairs_summary.csv")

print(OUTPUT_DIR / "strict_profile_group_statistics.csv")

In [ ]:

# ============================================================
# HPO ANNOTATION GAP AND CANDIDATE COVERAGE
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUT_DIR = Path("data/figures_for_paper")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATA
# ============================================================

total_diseases = 1276
hpo_annotated = 805
hpo_unannotated = 471

weighted_targets = 345
unweighted_targets = 345
strict_match_targets = 12

# Percentages for Panel A
annotated_pct = hpo_annotated / total_diseases * 100
unannotated_pct = hpo_unannotated / total_diseases * 100

# Percentages for Panel B, relative to HPO-unannotated diseases
weighted_pct = weighted_targets / hpo_unannotated * 100
unweighted_pct = unweighted_targets / hpo_unannotated * 100
strict_match_pct = strict_match_targets / hpo_unannotated * 100

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ============================================================
# PANEL A — HPO ANNOTATION COVERAGE
# ============================================================

ax = axes[0]

labels_a = [
    "Total diseases",
    "HPO-annotated",
    "HPO-unannotated"
]

values_a = [
    total_diseases,
    hpo_annotated,
    hpo_unannotated
]

bars_a = ax.bar(labels_a, values_a)

ax.set_title("A. HPO annotation coverage")
ax.set_ylabel("Number of diseases")
ax.set_ylim(0, max(values_a) * 1.20)

for bar, value, label in zip(bars_a, values_a, labels_a):
    height = bar.get_height()

    if label == "HPO-annotated":
        text = f"{value}\n({annotated_pct:.1f}%)"
    elif label == "HPO-unannotated":
        text = f"{value}\n({unannotated_pct:.1f}%)"
    else:
        text = f"{value}"

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        text,
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.tick_params(axis="x", rotation=25)

# ============================================================
# PANEL B — CANDIDATE COVERAGE AMONG HPO-UNANNOTATED DISEASES
# ============================================================

ax = axes[1]

labels_b = [
    "HPO-unannotated\ntargets",
    "Weighted\nsimilarity",
    "Unweighted\nsimilarity",
    "Strict\nmatches"
]

values_b = [
    hpo_unannotated,
    weighted_targets,
    unweighted_targets,
    strict_match_targets
]

bars_b = ax.bar(labels_b, values_b)

ax.set_title("B. Candidate coverage among HPO-unannotated diseases")
ax.set_ylabel("Number of HPO-unannotated diseases")
ax.set_ylim(0, max(values_b) * 1.25)

for bar, value, label in zip(bars_b, values_b, labels_b):
    height = bar.get_height()

    if "Weighted" in label:
        text = f"{value}\n({weighted_pct:.1f}%)"
    elif "Unweighted" in label:
        text = f"{value}\n({unweighted_pct:.1f}%)"
    elif "Strict" in label:
        text = f"{value}\n({strict_match_pct:.1f}%)"
    else:
        text = f"{value}\n(reference)"

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        text,
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.tick_params(axis="x", rotation=25)

# ============================================================
# LAYOUT AND SAVE FIGURES
# ============================================================

fig.suptitle(
    "HPO annotation gap and candidate coverage",
    fontsize=14,
    y=1.03
)

plt.tight_layout()

plt.savefig(
    OUT_DIR / "fig_hpo_annotation_gap_candidate_coverage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    OUT_DIR / "fig_hpo_annotation_gap_candidate_coverage.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
"""

Stage 2: Rare-disease dataset expansion and HPO prioritization

==============================================================

Implements the exact-match reaction-profile method described in the paper.



Corrections applied:

  A. Robust CSV reading: strip column names BEFORE selecting usecols.

  B. Complete-profile filter: dropna(how="any") so every Φ_j(d) has all 6 fields.

  C. HPO counted strictly by HPO_ID (not tuple with label) to avoid text-variant splitting.

  D. Supporting_ORPHAcode_donors exported per HPO term (= D_{h,ex}(d0)).

  E. is_selected and is_fully_supported flag columns added for clarity.

  F. Cofactor normalised as a sorted set so "Fe²⁺, BH4" == "BH4, Fe²⁺".

"""



from typing import Dict, List

import pandas as pd



# ---------------------------------------------------------------------------

# 0. Configuration

# ---------------------------------------------------------------------------



INPUT_FILE = "data/dataset_finale_Stoichiometry_CLEAN.csv"

OUTPUT_FILE = "data/hpo_prioritization_results.csv"



USECOLS = [

    "ORPHAcode", "DiseaseName", "Gene", "Entry",

    "Rhea_ID", "EC", "Cofactor", "Pathway",

    "HPO_ID", "HPO_Label",

]



PROFILE_KEYS = ["Gene", "Entry", "EC", "Rhea_ID", "Cofactor", "Pathway"]



# ---------------------------------------------------------------------------

# 1. Load data  — strip column names BEFORE selecting columns

# ---------------------------------------------------------------------------



print("Loading dataset …")

df = pd.read_csv(INPUT_FILE, sep=";", dtype=str, low_memory=False)

df.columns = df.columns.str.strip()

df = df[USECOLS].copy()



# Normalise every string cell

for col in df.columns:

    df[col] = df[col].str.strip()

    df[col] = df[col].replace(

        {

            "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "NAN": pd.NA,

            "none": pd.NA, "None": pd.NA, "NONE": pd.NA,

            "null": pd.NA, "Null": pd.NA, "NULL": pd.NA,

        }

    )



# ---------------------------------------------------------------------------

# Normalise Cofactor as a set (order-independent comparison).

#   "Fe²⁺, BH4"  and  "BH4, Fe²⁺"  →  "BH4, Fe²⁺"

# ---------------------------------------------------------------------------

def _normalise_cofactor(val):

    if pd.isna(val):

        return val

    parts = [p.strip() for p in val.split(",") if p.strip()]

    return ", ".join(sorted(parts))



df["Cofactor"] = df["Cofactor"].apply(_normalise_cofactor)



print(f"  Raw rows: {len(df):,}")



# ---------------------------------------------------------------------------

# Stage-2 methodological filter:

#   Exclude diseases that lack AT LEAST ONE cofactor OR AT LEAST ONE pathway.

# ---------------------------------------------------------------------------



diseases_with_cofactor = set(df.loc[df["Cofactor"].notna(), "ORPHAcode"])

diseases_with_pathway  = set(df.loc[df["Pathway"].notna(),  "ORPHAcode"])

valid_orpha_set        = diseases_with_cofactor & diseases_with_pathway



df = df[df["ORPHAcode"].isin(valid_orpha_set)].copy()

print(f"  Rows after Stage-2 cofactor/pathway filter: {len(df):,}")



# Canonical HPO_ID -> HPO_Label mapping (one label per ID, first wins)

hpo_label_map: Dict[str, str] = (

    df[["HPO_ID", "HPO_Label"]]

    .dropna(subset=["HPO_ID"])

    .drop_duplicates(subset=["HPO_ID"])

    .set_index("HPO_ID")["HPO_Label"]

    .to_dict()

)



# ---------------------------------------------------------------------------

# 2. Build reaction profiles P(d)

#    Each Φ_j(d) = (Gene, Entry, EC, Rhea_ID, Cofactor, Pathway).

#    Correction B: dropna(how="any") — all 6 fields required.

#    Correction F: Cofactor already normalised → order-invariant.

# ---------------------------------------------------------------------------



profile_df = (

    df[["ORPHAcode", "DiseaseName"] + PROFILE_KEYS]

    .drop_duplicates()

    .dropna(subset=PROFILE_KEYS, how="any")

)



def build_profile_set(group: pd.DataFrame) -> frozenset:

    """P(d) as a frozenset of fully-populated 6-tuples."""

    return frozenset(

        tuple(row[k] for k in PROFILE_KEYS)

        for _, row in group.iterrows()

    )



print("Building reaction profiles P(d) …")

disease_profiles = (

    profile_df

    .groupby("ORPHAcode")

    .apply(build_profile_set)

    .rename("profile_set")

    .reset_index()

)



disease_names    = df[["ORPHAcode", "DiseaseName"]].drop_duplicates("ORPHAcode")

disease_profiles = disease_profiles.merge(disease_names, on="ORPHAcode", how="left")



# ---------------------------------------------------------------------------

# 3. Build HPO sets H(d)

#    Correction C: set of HPO_ID strings only.

# ---------------------------------------------------------------------------



print("Building HPO sets H(d) …")



disease_hpo = (

    df[["ORPHAcode", "HPO_ID"]]

    .dropna(subset=["HPO_ID"])

    .drop_duplicates()

    .groupby("ORPHAcode")["HPO_ID"]

    .apply(set)

    .rename("hpo_set")

    .reset_index()

)



diseases = disease_profiles.merge(disease_hpo, on="ORPHAcode", how="left")

diseases["hpo_set"] = diseases["hpo_set"].apply(

    lambda x: x if isinstance(x, set) else set()

)

diseases["n_hpo"] = diseases["hpo_set"].apply(len)



print(f"  Total distinct valid diseases: {len(diseases):,}")



# ---------------------------------------------------------------------------

# 4. Classify: targets (H = ∅) vs donors (|H| > 0)

# ---------------------------------------------------------------------------



targets = diseases[diseases["n_hpo"] == 0].reset_index(drop=True)

donors  = diseases[diseases["n_hpo"] >  0].reset_index(drop=True)



print(f"  Target diseases  (H=∅):   {len(targets):,}")

print(f"  Donor  diseases (|H|>0): {len(donors):,}")



# ---------------------------------------------------------------------------

# 5. Exact reaction-profile matching

#    Index donors by frozenset for O(1) lookup per target.

# ---------------------------------------------------------------------------



print("Running exact reaction-profile matching …")



donor_profile_index: Dict[frozenset, List[int]] = {}

for idx, row in donors.iterrows():

    donor_profile_index.setdefault(row["profile_set"], []).append(idx)



results = []



for _, target in targets.iterrows():

    d0_code = target["ORPHAcode"]

    d0_name = target["DiseaseName"]

    d0_prof = target["profile_set"]



    matched_donors = donors.loc[donor_profile_index.get(d0_prof, [])]

    n_exact        = len(matched_donors)



    if n_exact == 0:

        results.append({

            "ORPHAcode_target":            d0_code,

            "DiseaseName_target":          d0_name,

            "n_exact_donors":              0,

            "ORPHAcode_donors":            "",

            "DiseaseName_donors":          "",

            "HPO_ID":                      "",

            "HPO_Label":                   "",

            "donor_count":                 0,

            "Supporting_ORPHAcode_donors": "",

            "candidate_type":              "no_match",

            "is_selected":                 False,

            "is_fully_supported":          False,

        })

        continue



    # -----------------------------------------------------------------------

    # 6. HPO prioritization

    #    Correction C: count by HPO_ID only.

    #    Correction D: track which donors carry each HPO_ID.

    # -----------------------------------------------------------------------



    hpo_donor_count: Dict[str, int]        = {}

    hpo_donor_codes: Dict[str, List[str]]  = {}



    for _, donor in matched_donors.iterrows():

        dk_code = str(donor["ORPHAcode"])

        for hpo_id in donor["hpo_set"]:

            hpo_donor_count[hpo_id] = hpo_donor_count.get(hpo_id, 0) + 1

            hpo_donor_codes.setdefault(hpo_id, []).append(dk_code)



    donor_codes_str = "|".join(matched_donors["ORPHAcode"].astype(str).tolist())

    donor_names_str = "|".join(matched_donors["DiseaseName"].fillna("").tolist())



    recurrent_counts  = [c for c in hpo_donor_count.values() if c >= 2]

    max_multi_support = max(recurrent_counts) if recurrent_counts else 0



    for hpo_id, count in hpo_donor_count.items():

        if count == 1:

            ctype = "single_donor"

        elif count >= 2:

            if count == max_multi_support:

                ctype = "fully_supported" if count == n_exact else "dominant"

            else:

                ctype = "recurrent"

        else:

            ctype = "single_donor"



        is_selected        = ctype in ("dominant", "fully_supported")

        is_fully_supported = ctype == "fully_supported"



        results.append({

            "ORPHAcode_target":            d0_code,

            "DiseaseName_target":          d0_name,

            "n_exact_donors":              n_exact,

            "ORPHAcode_donors":            donor_codes_str,

            "DiseaseName_donors":          donor_names_str,

            "HPO_ID":                      hpo_id,

            "HPO_Label":                   hpo_label_map.get(hpo_id, ""),

            "donor_count":                 count,

            "Supporting_ORPHAcode_donors": "|".join(hpo_donor_codes[hpo_id]),

            "candidate_type":              ctype,

            "is_selected":                 is_selected,

            "is_fully_supported":          is_fully_supported,

        })



# ---------------------------------------------------------------------------

# 7. Assemble, sort, and export

# ---------------------------------------------------------------------------



results_df = pd.DataFrame(results)



type_order = {

    "fully_supported": 0,

    "dominant":        1,

    "recurrent":       2,

    "single_donor":    3,

    "no_match":        4,

}

results_df["_type_order"] = results_df["candidate_type"].map(type_order)

results_df = (

    results_df

    .sort_values(

        ["ORPHAcode_target", "_type_order", "donor_count"],

        ascending=[True, True, False],

    )

    .drop(columns=["_type_order"])

    .reset_index(drop=True)

)



results_df.to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8-sig")



print(f"\nDone. Results written to:\n  {OUTPUT_FILE}")

print(f"  Total rows in output: {len(results_df):,}")



# ---------------------------------------------------------------------------

# 8. Summary

# ---------------------------------------------------------------------------



summary = results_df.groupby("candidate_type")["ORPHAcode_target"].nunique()

print("\nTarget diseases with >=1 candidate per type:")

print(summary.to_string())



matched  = results_df[results_df["candidate_type"] != "no_match"]["ORPHAcode_target"].nunique()

no_match = results_df[results_df["candidate_type"] == "no_match"]["ORPHAcode_target"].nunique()

print(f"\nTargets with >=1 exact donor match : {matched:,}")

print(f"Targets with no match             : {no_match:,}")


In [ ]:
"""

Sensitivity Analysis: cofactor/pathway filter ON vs OFF

=======================================================

Compares the Stage-2 pipeline with and without the methodological

filter that requires at least one Cofactor AND at least one Pathway

per disease.

"""



from typing import Dict, List, Tuple

import pandas as pd

import sys



INPUT_FILE = "data/dataset_finale_Stoichiometry_CLEAN.csv"



USECOLS = [

    "ORPHAcode", "DiseaseName", "Gene", "Entry",

    "Rhea_ID", "EC", "Cofactor", "Pathway",

    "HPO_ID", "HPO_Label",

]



PROFILE_KEYS = ["Gene", "Entry", "EC", "Rhea_ID", "Cofactor", "Pathway"]





def _normalise_cofactor(val):

    if pd.isna(val):

        return val

    parts = [p.strip() for p in val.split(",") if p.strip()]

    return ", ".join(sorted(parts))





def run_pipeline(use_cofactor_pathway_filter: bool) -> Tuple[pd.DataFrame, dict]:

    """

    Run the Stage-2 HPO prioritisation pipeline.



    Parameters

    ----------

    use_cofactor_pathway_filter : bool

        If True, only diseases with >=1 Cofactor AND >=1 Pathway are kept.

        If False, all diseases are retained (no methodological filter).



    Returns

    -------

    results_df : pd.DataFrame

        Full prioritisation output (same schema as hpo_prioritization_results.csv).

    stats : dict

        Summary statistics for comparison.

    """

    # ------------------------------------------------------------------

    # 1. Load data

    # ------------------------------------------------------------------

    df = pd.read_csv(INPUT_FILE, sep=";", dtype=str, low_memory=False)

    df.columns = df.columns.str.strip()

    df = df[USECOLS].copy()



    for col in df.columns:

        df[col] = df[col].str.strip()

        df[col] = df[col].replace(

            {

                "": pd.NA, "nan": pd.NA, "NaN": pd.NA, "NAN": pd.NA,

                "none": pd.NA, "None": pd.NA, "NONE": pd.NA,

                "null": pd.NA, "Null": pd.NA, "NULL": pd.NA,

            }

        )



    df["Cofactor"] = df["Cofactor"].apply(_normalise_cofactor)



    raw_rows = len(df)

    raw_diseases = df["ORPHAcode"].nunique()

    raw_diseases_with_hpo = df.dropna(subset=["HPO_ID"])["ORPHAcode"].nunique()



    # ------------------------------------------------------------------

    # Stage-2 methodological filter (optional)

    # ------------------------------------------------------------------

    if use_cofactor_pathway_filter:

        diseases_with_cofactor = set(df.loc[df["Cofactor"].notna(), "ORPHAcode"])

        diseases_with_pathway  = set(df.loc[df["Pathway"].notna(), "ORPHAcode"])

        valid_orpha_set        = diseases_with_cofactor & diseases_with_pathway

        df = df[df["ORPHAcode"].isin(valid_orpha_set)].copy()



    rows_after_filter = len(df)

    diseases_after_filter = df["ORPHAcode"].nunique()



    # ------------------------------------------------------------------

    # HPO label map

    # ------------------------------------------------------------------

    hpo_label_map: Dict[str, str] = (

        df[["HPO_ID", "HPO_Label"]]

        .dropna(subset=["HPO_ID"])

        .drop_duplicates(subset=["HPO_ID"])

        .set_index("HPO_ID")["HPO_Label"]

        .to_dict()

    )



    # ------------------------------------------------------------------

    # 2. Build reaction profiles P(d)  — requires all 6 fields

    # ------------------------------------------------------------------

    profile_df = (

        df[["ORPHAcode", "DiseaseName"] + PROFILE_KEYS]

        .drop_duplicates()

        .dropna(subset=PROFILE_KEYS, how="any")

    )



    def build_profile_set(group: pd.DataFrame) -> frozenset:

        return frozenset(

            tuple(row[k] for k in PROFILE_KEYS)

            for _, row in group.iterrows()

        )



    disease_profiles = (

        profile_df

        .groupby("ORPHAcode")

        .apply(build_profile_set)

        .rename("profile_set")

        .reset_index()

    )



    disease_names = df[["ORPHAcode", "DiseaseName"]].drop_duplicates("ORPHAcode")

    disease_profiles = disease_profiles.merge(disease_names, on="ORPHAcode", how="left")



    n_profiles = len(disease_profiles)



    # ------------------------------------------------------------------

    # 3. Build HPO sets H(d)

    # ------------------------------------------------------------------

    disease_hpo = (

        df[["ORPHAcode", "HPO_ID"]]

        .dropna(subset=["HPO_ID"])

        .drop_duplicates()

        .groupby("ORPHAcode")["HPO_ID"]

        .apply(set)

        .rename("hpo_set")

        .reset_index()

    )



    diseases = disease_profiles.merge(disease_hpo, on="ORPHAcode", how="left")

    diseases["hpo_set"] = diseases["hpo_set"].apply(

        lambda x: x if isinstance(x, set) else set()

    )

    diseases["n_hpo"] = diseases["hpo_set"].apply(len)



    # ------------------------------------------------------------------

    # 4. Classify targets vs donors

    # ------------------------------------------------------------------

    targets = diseases[diseases["n_hpo"] == 0].reset_index(drop=True)

    donors  = diseases[diseases["n_hpo"] >  0].reset_index(drop=True)



    n_targets = len(targets)

    n_donors  = len(donors)



    # ------------------------------------------------------------------

    # 5. Exact reaction-profile matching

    # ------------------------------------------------------------------

    donor_profile_index: Dict[frozenset, List[int]] = {}

    for idx, row in donors.iterrows():

        donor_profile_index.setdefault(row["profile_set"], []).append(idx)



    results = []



    for _, target in targets.iterrows():

        d0_code = target["ORPHAcode"]

        d0_name = target["DiseaseName"]

        d0_prof = target["profile_set"]



        matched_donors = donors.loc[donor_profile_index.get(d0_prof, [])]

        n_exact = len(matched_donors)



        if n_exact == 0:

            results.append({

                "ORPHAcode_target":            d0_code,

                "DiseaseName_target":          d0_name,

                "n_exact_donors":              0,

                "ORPHAcode_donors":            "",

                "DiseaseName_donors":          "",

                "HPO_ID":                      "",

                "HPO_Label":                   "",

                "donor_count":                 0,

                "Supporting_ORPHAcode_donors": "",

                "candidate_type":              "no_match",

                "is_selected":                 False,

                "is_fully_supported":          False,

            })

            continue



        # ------------------------------------------------------------------

        # 6. HPO prioritization

        # ------------------------------------------------------------------

        hpo_donor_count: Dict[str, int]        = {}

        hpo_donor_codes: Dict[str, List[str]]  = {}



        for _, donor in matched_donors.iterrows():

            dk_code = str(donor["ORPHAcode"])

            for hpo_id in donor["hpo_set"]:

                hpo_donor_count[hpo_id] = hpo_donor_count.get(hpo_id, 0) + 1

                hpo_donor_codes.setdefault(hpo_id, []).append(dk_code)



        donor_codes_str = "|".join(matched_donors["ORPHAcode"].astype(str).tolist())

        donor_names_str = "|".join(matched_donors["DiseaseName"].fillna("").tolist())



        recurrent_counts = [c for c in hpo_donor_count.values() if c >= 2]

        max_multi_support = max(recurrent_counts) if recurrent_counts else 0



        for hpo_id, count in hpo_donor_count.items():

            if count == 1:

                ctype = "single_donor"

            elif count >= 2:

                if count == max_multi_support:

                    ctype = "fully_supported" if count == n_exact else "dominant"

                else:

                    ctype = "recurrent"

            else:

                ctype = "single_donor"



            is_selected        = ctype in ("dominant", "fully_supported")

            is_fully_supported = ctype == "fully_supported"



            results.append({

                "ORPHAcode_target":            d0_code,

                "DiseaseName_target":          d0_name,

                "n_exact_donors":              n_exact,

                "ORPHAcode_donors":            donor_codes_str,

                "DiseaseName_donors":          donor_names_str,

                "HPO_ID":                      hpo_id,

                "HPO_Label":                   hpo_label_map.get(hpo_id, ""),

                "donor_count":                 count,

                "Supporting_ORPHAcode_donors": "|".join(hpo_donor_codes[hpo_id]),

                "candidate_type":              ctype,

                "is_selected":                 is_selected,

                "is_fully_supported":          is_fully_supported,

            })



    # ------------------------------------------------------------------

    # 7. Assemble output

    # ------------------------------------------------------------------

    results_df = pd.DataFrame(results)



    type_order = {

        "fully_supported": 0,

        "dominant":        1,

        "recurrent":       2,

        "single_donor":    3,

        "no_match":        4,

    }

    results_df["_type_order"] = results_df["candidate_type"].map(type_order)

    results_df = (

        results_df

        .sort_values(

            ["ORPHAcode_target", "_type_order", "donor_count"],

            ascending=[True, True, False],

        )

        .drop(columns=["_type_order"])

        .reset_index(drop=True)

    )



    # ------------------------------------------------------------------

    # 8. Compute summary statistics

    # ------------------------------------------------------------------

    n_matched  = results_df[results_df["candidate_type"] != "no_match"]["ORPHAcode_target"].nunique()

    n_no_match = results_df[results_df["candidate_type"] == "no_match"]["ORPHAcode_target"].nunique()



    # Per-type counts (target-level, not row-level)

    type_counts = {}

    for ct in ["fully_supported", "dominant", "recurrent", "single_donor", "no_match"]:

        candidates_ct = results_df[results_df["candidate_type"] == ct]

        type_counts[ct] = candidates_ct["ORPHAcode_target"].nunique() if len(candidates_ct) > 0 else 0



    # Candidate HPO terms per matched target

    matched_rows = results_df[results_df["candidate_type"] != "no_match"]

    candidates_per_target = matched_rows.groupby("ORPHAcode_target").size()

    mean_candidates = candidates_per_target.mean()

    median_candidates = candidates_per_target.median()

    min_candidates = candidates_per_target.min()

    max_candidates = candidates_per_target.max()



    # Total candidate HPO terms

    total_candidate_hpo = len(matched_rows)



    # Donors per matched target

    donors_per_target = matched_rows.groupby("ORPHAcode_target")["n_exact_donors"].first()

    mean_donors = donors_per_target.mean()



    stats = {

        "use_filter":                     use_cofactor_pathway_filter,

        "raw_rows":                       raw_rows,

        "raw_diseases":                   raw_diseases,

        "raw_diseases_with_hpo":          raw_diseases_with_hpo,

        "rows_after_filter":              rows_after_filter,

        "diseases_after_filter":          diseases_after_filter,

        "diseases_with_complete_profile": n_profiles,

        "n_targets":                      n_targets,

        "n_donors":                       n_donors,

        "n_matched":                      n_matched,

        "n_no_match":                     n_no_match,

        "type_counts":                    type_counts,

        "total_candidate_hpo":            total_candidate_hpo,

        "mean_candidates_per_target":     round(mean_candidates, 1),

        "median_candidates_per_target":   int(median_candidates),

        "min_candidates_per_target":      int(min_candidates),

        "max_candidates_per_target":      int(max_candidates),

        "mean_donors_per_matched_target": round(mean_donors, 1),

    }



    return results_df, stats





# ===================================================================

# MAIN: run both modes and print comparison

# ===================================================================

if __name__ == "__main__":

    print("=" * 72)

    print("Sensitivity analysis: cofactor/pathway filter ON vs OFF")

    print("=" * 72)



    all_results = {}

    all_stats   = {}



    for use_filter in [True, False]:

        label = "ON" if use_filter else "OFF"

        print(f"\n>>> Pipeline with filter = {label} ...\n")

        res, st = run_pipeline(use_filter)

        all_results[label] = res

        all_stats[label]   = st



    # ------------------------------------------------------------------

    # Print comparison table (plain text)

    # ------------------------------------------------------------------

    on  = all_stats["ON"]

    off = all_stats["OFF"]



    print("\n" + "=" * 72)

    print("COMPARISON SUMMARY")

    print("=" * 72)



    rows_comp = [

        ("Rows in input file",           f"{on['raw_rows']:,}",      f"{off['raw_rows']:,}",      "---"),

        ("Distinct diseases (raw)",      f"{on['raw_diseases']:,}",  f"{off['raw_diseases']:,}",  "---"),

        ("After Cof/Path filter",        f"{on['diseases_after_filter']:,}", f"{off['diseases_after_filter']:,}", "---"),

        ("With complete 6-field profile",f"{on['diseases_with_complete_profile']:,}", f"{off['diseases_with_complete_profile']:,}", f"{off['diseases_with_complete_profile'] - on['diseases_with_complete_profile']:+}"),

        ("",                             "",                          "",                          ""),

        ("Targets (H=empty)",            f"{on['n_targets']:,}",     f"{off['n_targets']:,}",     f"{off['n_targets'] - on['n_targets']:+}"),

        ("Donors (|H|>0)",               f"{on['n_donors']:,}",      f"{off['n_donors']:,}",      f"{off['n_donors'] - on['n_donors']:+}"),

        ("",                             "",                          "",                          ""),

        ("Matched targets (exact match)",f"{on['n_matched']:,}",     f"{off['n_matched']:,}",     f"{off['n_matched'] - on['n_matched']:+}"),

        ("No-match targets",             f"{on['n_no_match']:,}",    f"{off['n_no_match']:,}",    f"{off['n_no_match'] - on['n_no_match']:+}"),

        ("",                             "",                          "",                          ""),

    ]



    # Add per-type counts

    for ct in ["fully_supported", "dominant", "recurrent", "single_donor", "no_match"]:

        rows_comp.append(

            (f"  Targets with {ct} candidates",

             f"{on['type_counts'][ct]:,}",

             f"{off['type_counts'][ct]:,}",

             f"{off['type_counts'][ct] - on['type_counts'][ct]:+}")

        )



    rows_comp += [

        ("",                             "",                          "",                          ""),

        ("Total candidate HPO terms",    f"{on['total_candidate_hpo']:,}", f"{off['total_candidate_hpo']:,}", f"{off['total_candidate_hpo'] - on['total_candidate_hpo']:+}"),

        ("Mean candidates / matched target", f"{on['mean_candidates_per_target']}", f"{off['mean_candidates_per_target']}", "---"),

        ("Median candidates / matched target", f"{on['median_candidates_per_target']}", f"{off['median_candidates_per_target']}", "---"),

        ("Min / Max candidates",         f"{on['min_candidates_per_target']}/{on['max_candidates_per_target']}", f"{off['min_candidates_per_target']}/{off['max_candidates_per_target']}", "---"),

        ("Mean donors / matched target", f"{on['mean_donors_per_matched_target']}", f"{off['mean_donors_per_matched_target']}", "---"),

    ]



    header = f"{'Metric':<55} {'Filter ON':>12} {'Filter OFF':>12} {'Delta':>8}"

    sep = "-" * len(header)

    print(f"\n{header}")

    print(sep)

    for label, v_on, v_off, delta in rows_comp:

        if not label:

            print()

        else:

            print(f"{label:<55} {v_on:>12} {v_off:>12} {delta:>8}")



    # ------------------------------------------------------------------

    # LaTeX table for the paper

    # ------------------------------------------------------------------

    print("\n" + "=" * 72)

    print("LATEX TABLE (copy-paste into paper)")

    print("=" * 72)



    # Helper to format delta with sign

    def fmt_delta(val_on, val_off):

        d = val_off - val_on

        if d > 0:

            return f"$+{d}$"

        elif d < 0:

            return f"${d}$"

        else:

            return "$0$"



    print("""

\\\begin{table}[t]

\\\centering

\\\caption{Sensitivity analysis of the cofactor/pathway filter.

         Filter~ON requires each disease to have at least one annotated

         cofactor and at least one pathway; Filter~OFF omits this

         requirement.  The 6-field reaction profile

         (Gene, Entry, EC, Rhea\textunderscore ID, Cofactor, Pathway)

         and complete-profile requirement ($\\\Phi_j(d)$ non-null for all

         six fields) are applied in both settings.}

\\\label{tab:sensitivity}

\\\small

\\\begin{tabular}{lrrr}

\\\toprule

\\\textbf{Metric} & \\\textbf{Filter ON} & \\\textbf{Filter OFF} & $\\\bm{\\\Delta}$ \\\\\\\\

\\\midrule""")



    latex_rows = [

        ("Diseases with complete 6-field profile",

         f"{on['diseases_with_complete_profile']:,}",

         f"{off['diseases_with_complete_profile']:,}",

         fmt_delta(on['diseases_with_complete_profile'], off['diseases_with_complete_profile'])),

        ("Target diseases ($H(d)=\\\\\\\emptyset$)",

         f"{on['n_targets']:,}",

         f"{off['n_targets']:,}",

         fmt_delta(on['n_targets'], off['n_targets'])),

        ("Donor diseases ($|H(d)|>0$)",

         f"{on['n_donors']:,}",

         f"{off['n_donors']:,}",

         fmt_delta(on['n_donors'], off['n_donors'])),

        ("\\\midrule"),

        ("Matched targets (exact reaction-profile match)",

         f"{on['n_matched']:,}",

         f"{off['n_matched']:,}",

         fmt_delta(on['n_matched'], off['n_matched'])),

        ("No-match targets",

         f"{on['n_no_match']:,}",

         f"{off['n_no_match']:,}",

         fmt_delta(on['n_no_match'], off['n_no_match'])),

        ("\\\midrule"),

        ("Total candidate HPO terms",

         f"{on['total_candidate_hpo']:,}",

         f"{off['total_candidate_hpo']:,}",

         fmt_delta(on['total_candidate_hpo'], off['total_candidate_hpo'])),

    ]



    for ct_label, ct_key in [("fully\\\\_supported", "fully_supported"),

                              ("dominant", "dominant"),

                              ("recurrent", "recurrent"),

                              ("single\\\\_donor", "single_donor")]:

        latex_rows.append(

            (f"~~Targets with {ct_label} candidates",

             f"{on['type_counts'][ct_key]:,}",

             f"{off['type_counts'][ct_key]:,}",

             fmt_delta(on['type_counts'][ct_key], off['type_counts'][ct_key]))

        )



    latex_rows += [

        ("\\\midrule"),

        ("Mean candidates per matched target",

         str(on['mean_candidates_per_target']),

         str(off['mean_candidates_per_target']),

         "---"),

        ("Median candidates per matched target",

         str(on['median_candidates_per_target']),

         str(off['median_candidates_per_target']),

         "---"),

        ("Mean donor diseases per matched target",

         str(on['mean_donors_per_matched_target']),

         str(off['mean_donors_per_matched_target']),

         "---"),

    ]



    for parts in latex_rows:

        if isinstance(parts, str) and parts.startswith("\\\midrule"):

            print("\\\midrule")

        elif isinstance(parts, tuple) and parts[0].startswith("\\\midrule"):

            print("\\\midrule")

        else:

            lbl, v_on, v_off, d = parts

            print(f"{lbl} & {v_on} & {v_off} & {d} \\\\\\\\")



    print("""\\\bottomrule

\\\end{tabular}

\\\end{table}""")



    # ------------------------------------------------------------------

    # Also save both outputs to CSV for inspection

    # ------------------------------------------------------------------

    out_on = "data/hpo_prioritization_filter_ON.csv"

    out_off = "data/hpo_prioritization_filter_OFF.csv"

    all_results["ON"].to_csv(out_on,  sep=";", index=False, encoding="utf-8-sig")

    all_results["OFF"].to_csv(out_off, sep=";", index=False, encoding="utf-8-sig")

    print(f"\nDetailed results saved to:")

    print(f"  Filter ON : {out_on}")

    print(f"  Filter OFF: {out_off}")

    print("\nDone.")


In [ ]:
"""
Baseline comparison: matching criteria from simple (Gene-only) to
complete (6-field). Runs exact-set matching on the same 117-disease
pool and reports matched targets + mean candidates per target.
"""

from typing import Dict, List, Tuple
import pandas as pd

INPUT_FILE = "data/dataset_finale_Stoichiometry_CLEAN.csv"

USECOLS = [
    "ORPHAcode", "DiseaseName", "Gene", "Entry",
    "Rhea_ID", "EC", "Cofactor", "Pathway",
    "HPO_ID", "HPO_Label",
]

FULL_PROFILE = ["Gene", "Entry", "EC", "Rhea_ID", "Cofactor", "Pathway"]

def _normalise_cofactor(val):
    if pd.isna(val):
        return val
    parts = [p.strip() for p in val.split(",") if p.strip()]
    return ", ".join(sorted(parts))

def load_and_filter() -> pd.DataFrame:
    """Load, normalise, apply cofactor/pathway filter, return clean df."""
    df = pd.read_csv(INPUT_FILE, sep=";", dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].str.strip()
        df[col] = df[col].replace(
            {"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "NAN": pd.NA,
             "none": pd.NA, "None": pd.NA, "NONE": pd.NA,
             "null": pd.NA, "Null": pd.NA, "NULL": pd.NA}
        )
    df["Cofactor"] = df["Cofactor"].apply(_normalise_cofactor)
    # filter
    cof = set(df.loc[df["Cofactor"].notna(), "ORPHAcode"])
    pth = set(df.loc[df["Pathway"].notna(),   "ORPHAcode"])
    df  = df[df["ORPHAcode"].isin(cof & pth)].copy()
    return df


def run_matching(profile_keys: List[str], df: pd.DataFrame) -> dict:
    """
    Run exact-set matching using only `profile_keys` for profile
    construction.  The 6-field completeness requirement (dropna on
    FULL_PROFILE) is always enforced so the disease pool is identical.
    """
    # ---- HPO label map ----
    hpo_label_map: Dict[str, str] = (
        df[["HPO_ID", "HPO_Label"]]
        .dropna(subset=["HPO_ID"])
        .drop_duplicates(subset=["HPO_ID"])
        .set_index("HPO_ID")["HPO_Label"]
        .to_dict()
    )

    # ---- Profiles (always complete on FULL_PROFILE) ----
    profile_df = (
        df[["ORPHAcode", "DiseaseName"] + FULL_PROFILE]
        .drop_duplicates()
        .dropna(subset=FULL_PROFILE, how="any")
    )

    # Build tuple using ONLY the requested keys
    def build_profile_set(group: pd.DataFrame) -> frozenset:
        return frozenset(
            tuple(row[k] for k in profile_keys)
            for _, row in group.iterrows()
        )

    disease_profiles = (
        profile_df[["ORPHAcode"] + profile_keys]
        .drop_duplicates()
        .groupby("ORPHAcode", group_keys=False)
        .apply(build_profile_set)
        .rename("profile_set")
        .reset_index()
    )

    disease_names = df[["ORPHAcode", "DiseaseName"]].drop_duplicates("ORPHAcode")
    disease_profiles = disease_profiles.merge(disease_names, on="ORPHAcode", how="left")

    n_profiles = len(disease_profiles)

    # ---- HPO sets ----
    disease_hpo = (
        df[["ORPHAcode", "HPO_ID"]]
        .dropna(subset=["HPO_ID"])
        .drop_duplicates()
        .groupby("ORPHAcode")["HPO_ID"]
        .apply(set)
        .rename("hpo_set")
        .reset_index()
    )

    diseases = disease_profiles.merge(disease_hpo, on="ORPHAcode", how="left")
    diseases["hpo_set"] = diseases["hpo_set"].apply(
        lambda x: x if isinstance(x, set) else set()
    )
    diseases["n_hpo"] = diseases["hpo_set"].apply(len)

    # ---- Split ----
    targets = diseases[diseases["n_hpo"] == 0].reset_index(drop=True)
    donors  = diseases[diseases["n_hpo"] >  0].reset_index(drop=True)
    n_targets = len(targets)
    n_donors  = len(donors)

    # ---- Matching ----
    donor_index: Dict[frozenset, List[int]] = {}
    for idx, row in donors.iterrows():
        donor_index.setdefault(row["profile_set"], []).append(idx)

    results = []
    for _, target in targets.iterrows():
        matched = donors.loc[donor_index.get(target["profile_set"], [])]
        n_match = len(matched)
        if n_match == 0:
            results.append({"n_exact": 0, "n_candidates": 0})
            continue

        hpo_counts: Dict[str, int] = {}
        for _, donor in matched.iterrows():
            for hpo_id in donor["hpo_set"]:
                hpo_counts[hpo_id] = hpo_counts.get(hpo_id, 0) + 1

        results.append({"n_exact": n_match, "n_candidates": len(hpo_counts)})

    # ---- Stats ----
    df_res = pd.DataFrame(results)
    n_matched  = (df_res["n_exact"] > 0).sum()
    n_no_match = (df_res["n_exact"] == 0).sum()
    matched_rows = df_res[df_res["n_exact"] > 0]
    mean_candidates = matched_rows["n_candidates"].mean() if len(matched_rows) else 0
    mean_donors     = matched_rows["n_exact"].mean() if len(matched_rows) else 0

    return {
        "label":            " + ".join(profile_keys),
        "n_profiles":       n_profiles,
        "n_targets":        n_targets,
        "n_donors":         n_donors,
        "n_matched":        n_matched,
        "n_no_match":       n_no_match,
        "total_candidates": int(df_res["n_candidates"].sum()),
        "mean_candidates":  round(mean_candidates, 1),
        "mean_donors":      round(mean_donors, 1),
    }


# ===================================================================
if __name__ == "__main__":
    print("Loading and filtering data ...")
    df = load_and_filter()

    # Baseline combinations for MAIN PAPER
    MAIN_BASELINES = [
        ["Gene"],
        ["Entry"],
        ["EC"],
        ["Rhea_ID"],
        ["Pathway"],
        ["Gene", "EC", "Rhea_ID"],
        ["Gene", "Entry", "EC", "Rhea_ID"],
        ["Gene", "Entry", "EC", "Rhea_ID", "Cofactor", "Pathway"],   # 6-field
    ]

    MAIN_LABELS = [
        "Gene-only",
        "Entry-only",
        "EC-only",
        "Rhea-only",
        "Pathway-only",
        "Gene+EC+Rhea",
        "Gene+Entry+EC+Rhea",
        "Exact 6-field",
    ]

    print(f"\n{'Criterion':<30} {'Matched':>8} {'Mean cand.':>10} {'Mean donors':>11}")
    print("-" * 62)

    all_stats = []
    for keys, lbl in zip(MAIN_BASELINES, MAIN_LABELS):
        st = run_matching(keys, df)
        st["display_label"] = lbl
        all_stats.append(st)
        print(f"{lbl:<30} {st['n_matched']:>3}/{st['n_targets']}     {st['mean_candidates']:>8.1f}     {st['mean_donors']:>8.1f}")

    # ------------------------------------------------------------------
    # LaTeX table
    # ------------------------------------------------------------------
    print("\n" + "=" * 62)
    print("LATEX TABLE (main paper)")
    print("=" * 62)

    print("""
\\begin{table*}[!t]
\\centering
\\caption{Internal comparison of biologically motivated matching criteria.
         Each row reports the number of matched targets (out of 32
         unannotated targets) and the mean candidate HPO terms per
         matched target.  All criteria use the same 117-disease pool
         with complete six-field profiles.}
\\label{tab:baseline_comparison}
\\small
\\begin{tabular}{lccc}
\\toprule
\\textbf{Matching criterion} & \\textbf{Matched targets} &
\\textbf{Mean candidates} & \\textbf{Mean donors} \\\\
\\midrule""")

    for st in all_stats:
        label = st["display_label"]
        matched_str = f"{st['n_matched']}/32"
        print(f"{label:40} & {matched_str:>6}      & {st['mean_candidates']:>6.1f}        & {st['mean_donors']:>5.1f} \\\\")

    print("""\\bottomrule
\\end{tabular}
\\end{table*}""")

    # ------------------------------------------------------------------
    # Supplementary baselines
    # ------------------------------------------------------------------
    SUPP_BASELINES = [
        ["Gene", "EC"],
        ["Gene", "Rhea_ID"],
        ["EC", "Rhea_ID"],
        ["Gene", "Entry"],
        ["Gene", "Entry", "EC", "Rhea_ID", "Cofactor"],
        ["Gene", "Entry", "EC", "Rhea_ID", "Pathway"],
    ]
    SUPP_LABELS = [
        "Gene+EC",
        "Gene+Rhea",
        "EC+Rhea",
        "Gene+Entry",
        "Gene+Entry+EC+Rhea+Cofactor",
        "Gene+Entry+EC+Rhea+Pathway",
    ]

    print("\n" + "=" * 62)
    print("SUPPLEMENTARY BASELINES")
    print("=" * 62)
    print(f"{'Criterion':<40} {'Matched':>8} {'Mean cand.':>10} {'Mean donors':>11}")
    print("-" * 72)
    for keys, lbl in zip(SUPP_BASELINES, SUPP_LABELS):
        st = run_matching(keys, df)
        print(f"{lbl:<40} {st['n_matched']:>3}/{st['n_targets']}     {st['mean_candidates']:>8.1f}     {st['mean_donors']:>8.1f}")

    print("\nDone.")


In [ ]:
# -*- coding: utf-8 -*-
"""
Stage 2: Rare-disease dataset expansion and HPO prioritization
==============================================================

Implements exact reaction-profile matching between:

    unannotated_diseases:
        diseases with a complete biochemical profile but no HPO annotation;

    annotated_diseases:
        diseases with the same type of biochemical profile and at least
        one HPO annotation.

For every unannotated disease, the script identifies annotated diseases
having an identical reaction profile and prioritizes their HPO terms.

Compatibility
-------------
This version does not use:

    include_groups=False

and is therefore compatible with pandas versions older than 2.2.

Corrections applied
-------------------
A. CSV column names are stripped before selecting USECOLS.
B. Every biochemical profile tuple must contain all six fields.
C. HPO terms are counted using HPO_ID only.
D. Supporting ORPHA codes are exported for each HPO candidate.
E. Selection and full-support flags are exported.
F. Cofactors are normalized as order-independent sets.
G. Naming:
       targets       -> unannotated_diseases
       donors        -> annotated_diseases
       single_donor  -> single_match
"""

from typing import Dict, List

import pandas as pd


# ===========================================================================
# 0. CONFIGURATION
# ===========================================================================

INPUT_FILE = "data/dataset_finale_Stoichiometry_CLEAN.csv"

OUTPUT_FILE = (
    r"C:\Users\biofa\Desktop\DATAEXTRACTION"
    r"\hpo_prioritization_results.csv"
)


USECOLS = [
    "ORPHAcode",
    "DiseaseName",
    "Gene",
    "Entry",
    "Rhea_ID",
    "EC",
    "Cofactor",
    "Pathway",
    "HPO_ID",
    "HPO_Label",
]


# Ordered fields defining one biochemical reaction tuple Phi_j(d)
PROFILE_KEYS = [
    "Gene",
    "Entry",
    "EC",
    "Rhea_ID",
    "Cofactor",
    "Pathway",
]


# Columns exported in the final CSV.
# Declaring them explicitly also prevents errors if no result is generated.
OUTPUT_COLUMNS = [
    "ORPHAcode_target",
    "DiseaseName_target",
    "n_exact_donors",
    "ORPHAcode_donors",
    "DiseaseName_donors",
    "HPO_ID",
    "HPO_Label",
    "donor_count",
    "Supporting_ORPHAcode_donors",
    "candidate_type",
    "is_selected",
    "is_fully_supported",
]


# ===========================================================================
# 1. HELPER FUNCTIONS
# ===========================================================================

def normalise_cofactor(value):
    """
    Normalize a cofactor field as an order-independent set.

    Examples
    --------
    "Fe²⁺, BH4" -> "BH4, Fe²⁺"
    "BH4, Fe²⁺" -> "BH4, Fe²⁺"
    """
    if pd.isna(value):
        return value

    parts = [
        part.strip()
        for part in str(value).split(",")
        if part.strip()
    ]

    # set() also removes accidental repeated cofactors.
    return ", ".join(sorted(set(parts)))


def build_profile_set(group: pd.DataFrame) -> frozenset:
    """
    Build the biochemical profile P(d) for one disease.

    The input contains only PROFILE_KEYS because the columns are explicitly
    selected before GroupBy.apply(). Therefore, ORPHAcode is never included
    in the biochemical tuple.

    Returns
    -------
    frozenset
        Set of complete six-field biochemical reaction tuples.
    """
    return frozenset(
        group.itertuples(index=False, name=None)
    )


# ===========================================================================
# 2. LOAD AND CLEAN THE DATASET
# ===========================================================================

print("=" * 72)
print("STAGE 2: REACTION-PROFILE MATCHING AND HPO PRIORITIZATION")
print("=" * 72)

print("\nLoading dataset ...")

df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    dtype=str,
    low_memory=False,
)


# Strip spaces from column names before selecting USECOLS.
df.columns = df.columns.str.strip()


# Check that all required columns exist.
missing_columns = [
    column
    for column in USECOLS
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The input file is missing the following required columns: "
        + ", ".join(missing_columns)
    )


df = df[USECOLS].copy()


# Normalize whitespace and missing-value representations.
missing_value_strings = {
    "": pd.NA,
    "nan": pd.NA,
    "NaN": pd.NA,
    "NAN": pd.NA,
    "none": pd.NA,
    "None": pd.NA,
    "NONE": pd.NA,
    "null": pd.NA,
    "Null": pd.NA,
    "NULL": pd.NA,
}


for column in df.columns:
    df[column] = df[column].str.strip()
    df[column] = df[column].replace(missing_value_strings)


# Rows without an ORPHA code cannot be assigned to a disease.
df = df.dropna(subset=["ORPHAcode"]).copy()


# Normalize cofactors before profile construction.
df["Cofactor"] = df["Cofactor"].apply(normalise_cofactor)


print("  Raw valid rows: %d" % len(df))
print("  Raw distinct diseases: %d" % df["ORPHAcode"].nunique())


# ===========================================================================
# 3. STAGE-2 COFACTOR/PATHWAY FILTER
# ===========================================================================

print("\nApplying the Stage-2 cofactor/pathway filter ...")


# A disease is retained only if it has:
#   1. at least one non-null Cofactor value;
#   2. at least one non-null Pathway value.
diseases_with_cofactor = set(
    df.loc[
        df["Cofactor"].notna(),
        "ORPHAcode",
    ]
)

diseases_with_pathway = set(
    df.loc[
        df["Pathway"].notna(),
        "ORPHAcode",
    ]
)

valid_orpha_set = (
    diseases_with_cofactor
    & diseases_with_pathway
)


df = df[
    df["ORPHAcode"].isin(valid_orpha_set)
].copy()


print(
    "  Rows after cofactor/pathway filter: %d"
    % len(df)
)

print(
    "  Diseases after cofactor/pathway filter: %d"
    % df["ORPHAcode"].nunique()
)


# ===========================================================================
# 4. BUILD THE CANONICAL HPO LABEL MAP
# ===========================================================================

# One canonical label is retained for every HPO identifier.
# If textual label variants exist, the first available value is used.
hpo_label_map: Dict[str, str] = (
    df[
        ["HPO_ID", "HPO_Label"]
    ]
    .dropna(subset=["HPO_ID"])
    .drop_duplicates(subset=["HPO_ID"])
    .set_index("HPO_ID")["HPO_Label"]
    .fillna("")
    .to_dict()
)


# ===========================================================================
# 5. BUILD BIOCHEMICAL REACTION PROFILES P(d)
# ===========================================================================

print("\nBuilding biochemical reaction profiles P(d) ...")


# Keep only complete six-field reaction tuples.
profile_df = (
    df[
        ["ORPHAcode", "DiseaseName"]
        + PROFILE_KEYS
    ]
    .drop_duplicates()
    .dropna(
        subset=PROFILE_KEYS,
        how="any",
    )
)


# ---------------------------------------------------------------------------
# IMPORTANT PANDAS COMPATIBILITY FIX
#
# PROFILE_KEYS are selected before apply().
# Therefore build_profile_set() receives only the six biochemical columns.
#
# No include_groups=False argument is used.
# ---------------------------------------------------------------------------

disease_profiles = (
    profile_df
    .groupby(
        "ORPHAcode",
        sort=False,
    )[PROFILE_KEYS]
    .apply(build_profile_set)
    .rename("profile_set")
    .reset_index()
)


# Add one disease name for each ORPHA code.
disease_names = (
    df[
        ["ORPHAcode", "DiseaseName"]
    ]
    .drop_duplicates(subset=["ORPHAcode"])
)


disease_profiles = disease_profiles.merge(
    disease_names,
    on="ORPHAcode",
    how="left",
)


print(
    "  Diseases with a complete biochemical profile: %d"
    % len(disease_profiles)
)


# ===========================================================================
# 6. BUILD HPO SETS H(d)
# ===========================================================================

print("\nBuilding HPO annotation sets H(d) ...")


# HPO terms are represented by identifiers only.
# Labels are added later using hpo_label_map.
disease_hpo = (
    df[
        ["ORPHAcode", "HPO_ID"]
    ]
    .dropna(subset=["HPO_ID"])
    .drop_duplicates()
    .groupby("ORPHAcode")["HPO_ID"]
    .apply(set)
    .rename("hpo_set")
    .reset_index()
)


# Keep every disease that has a complete biochemical profile,
# including diseases without HPO annotations.
diseases = disease_profiles.merge(
    disease_hpo,
    on="ORPHAcode",
    how="left",
)


# Diseases without HPO annotations receive an empty set.
diseases["hpo_set"] = diseases["hpo_set"].apply(
    lambda value: value
    if isinstance(value, set)
    else set()
)


diseases["n_hpo"] = diseases["hpo_set"].apply(len)


print(
    "  Total valid diseases: %d"
    % len(diseases)
)


# ===========================================================================
# 7. CLASSIFY UNANNOTATED AND ANNOTATED DISEASES
# ===========================================================================

# Unannotated diseases:
# complete biochemical profile, but H(d) is empty.
unannotated_diseases = (
    diseases[
        diseases["n_hpo"] == 0
    ]
    .reset_index(drop=True)
)


# Annotated diseases:
# complete biochemical profile and at least one HPO term.
annotated_diseases = (
    diseases[
        diseases["n_hpo"] > 0
    ]
    .reset_index(drop=True)
)


print(
    "  Unannotated diseases, H(d)=empty: %d"
    % len(unannotated_diseases)
)

print(
    "  Annotated diseases, |H(d)|>0:   %d"
    % len(annotated_diseases)
)


# ===========================================================================
# 8. INDEX ANNOTATED DISEASES BY EXACT BIOCHEMICAL PROFILE
# ===========================================================================

print("\nIndexing annotated diseases by biochemical profile ...")


# Dictionary structure:
#
#     biochemical profile -> row indices in annotated_diseases
#
# Because profile_set is a frozenset, it can be used as a dictionary key.
annotated_profile_index: Dict[frozenset, List[int]] = {}


for row_index, row in annotated_diseases.iterrows():
    profile = row["profile_set"]

    annotated_profile_index.setdefault(
        profile,
        [],
    ).append(row_index)


print(
    "  Distinct annotated biochemical profiles: %d"
    % len(annotated_profile_index)
)


# ===========================================================================
# 9. EXACT PROFILE MATCHING AND HPO PRIORITIZATION
# ===========================================================================

print("\nRunning exact biochemical-profile matching ...")


results = []


for _, unannotated_disease in unannotated_diseases.iterrows():

    target_orpha_code = unannotated_disease["ORPHAcode"]
    target_name = unannotated_disease["DiseaseName"]
    target_profile = unannotated_disease["profile_set"]


    # Retrieve annotated diseases having exactly the same frozenset profile.
    matching_indices = annotated_profile_index.get(
        target_profile,
        [],
    )


    matched_annotated_diseases = annotated_diseases.loc[
        matching_indices
    ]


    n_exact_matches = len(
        matched_annotated_diseases
    )


    # -----------------------------------------------------------------------
    # No annotated disease has the same biochemical profile.
    # -----------------------------------------------------------------------

    if n_exact_matches == 0:

        results.append(
            {
                "ORPHAcode_target": target_orpha_code,
                "DiseaseName_target": target_name,
                "n_exact_donors": 0,
                "ORPHAcode_donors": "",
                "DiseaseName_donors": "",
                "HPO_ID": "",
                "HPO_Label": "",
                "donor_count": 0,
                "Supporting_ORPHAcode_donors": "",
                "candidate_type": "no_match",
                "is_selected": False,
                "is_fully_supported": False,
            }
        )

        continue


    # -----------------------------------------------------------------------
    # Count support for each HPO term.
    #
    # hpo_match_count:
    #     number of matched annotated diseases carrying the HPO term.
    #
    # hpo_matching_orpha_codes:
    #     exact annotated diseases supporting that HPO term.
    # -----------------------------------------------------------------------

    hpo_match_count: Dict[str, int] = {}

    hpo_matching_orpha_codes: Dict[
        str,
        List[str],
    ] = {}


    for _, annotated_disease in matched_annotated_diseases.iterrows():

        annotated_orpha_code = str(
            annotated_disease["ORPHAcode"]
        )


        for hpo_id in annotated_disease["hpo_set"]:

            hpo_match_count[hpo_id] = (
                hpo_match_count.get(hpo_id, 0)
                + 1
            )

            hpo_matching_orpha_codes.setdefault(
                hpo_id,
                [],
            ).append(annotated_orpha_code)


    # All exact matched annotated diseases for this target.
    matched_orpha_codes_string = "|".join(
        matched_annotated_diseases[
            "ORPHAcode"
        ]
        .astype(str)
        .tolist()
    )


    matched_disease_names_string = "|".join(
        matched_annotated_diseases[
            "DiseaseName"
        ]
        .fillna("")
        .astype(str)
        .tolist()
    )


    # Maximum support among terms occurring in at least two matches.
    recurrent_support_values = [
        count
        for count in hpo_match_count.values()
        if count >= 2
    ]


    maximum_recurrent_support = (
        max(recurrent_support_values)
        if recurrent_support_values
        else 0
    )


    # -----------------------------------------------------------------------
    # Candidate classification
    # -----------------------------------------------------------------------

    for hpo_id, support_count in hpo_match_count.items():

        if support_count == 1:

            candidate_type = "single_match"


        elif support_count >= 2:

            if support_count == maximum_recurrent_support:

                if support_count == n_exact_matches:
                    candidate_type = "fully_supported"

                else:
                    candidate_type = "dominant"

            else:
                candidate_type = "recurrent"


        else:
            # Defensive fallback; normally impossible.
            candidate_type = "single_match"


        is_selected = candidate_type in {
            "dominant",
            "fully_supported",
        }


        is_fully_supported = (
            candidate_type == "fully_supported"
        )


        results.append(
            {
                "ORPHAcode_target":
                    target_orpha_code,

                "DiseaseName_target":
                    target_name,

                "n_exact_donors":
                    n_exact_matches,

                "ORPHAcode_donors":
                    matched_orpha_codes_string,

                "DiseaseName_donors":
                    matched_disease_names_string,

                "HPO_ID":
                    hpo_id,

                "HPO_Label":
                    hpo_label_map.get(hpo_id, ""),

                "donor_count":
                    support_count,

                "Supporting_ORPHAcode_donors":
                    "|".join(
                        hpo_matching_orpha_codes[hpo_id]
                    ),

                "candidate_type":
                    candidate_type,

                "is_selected":
                    is_selected,

                "is_fully_supported":
                    is_fully_supported,
            }
        )


# ===========================================================================
# 10. ASSEMBLE AND SORT THE OUTPUT
# ===========================================================================

results_df = pd.DataFrame(
    results,
    columns=OUTPUT_COLUMNS,
)


type_order = {
    "fully_supported": 0,
    "dominant": 1,
    "recurrent": 2,
    "single_match": 3,
    "no_match": 4,
}


if not results_df.empty:

    results_df["_type_order"] = (
        results_df["candidate_type"]
        .map(type_order)
        .fillna(99)
    )


    results_df = (
        results_df
        .sort_values(
            by=[
                "ORPHAcode_target",
                "_type_order",
                "donor_count",
                "HPO_ID",
            ],
            ascending=[
                True,
                True,
                False,
                True,
            ],
            na_position="last",
        )
        .drop(columns=["_type_order"])
        .reset_index(drop=True)
    )


# ===========================================================================
# 11. EXPORT RESULTS
# ===========================================================================

results_df.to_csv(
    OUTPUT_FILE,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)


print("\nResults written to:")
print("  %s" % OUTPUT_FILE)

print(
    "  Total output rows: %d"
    % len(results_df)
)


# ===========================================================================
# 12. SUMMARY
# ===========================================================================

print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)


print(
    "Unannotated diseases examined: %d"
    % len(unannotated_diseases)
)

print(
    "Annotated diseases available:  %d"
    % len(annotated_diseases)
)


if results_df.empty:

    print(
        "\nNo output rows were generated. "
        "Check whether the dataset contains diseases "
        "with complete biochemical profiles."
    )

else:

    summary = (
        results_df
        .groupby("candidate_type")[
            "ORPHAcode_target"
        ]
        .nunique()
        .reindex(
            [
                "fully_supported",
                "dominant",
                "recurrent",
                "single_match",
                "no_match",
            ],
            fill_value=0,
        )
    )


    print(
        "\nNumber of unannotated diseases "
        "with candidates in each class:"
    )

    print(summary.to_string())


    matched_targets = (
        results_df.loc[
            results_df["candidate_type"] != "no_match",
            "ORPHAcode_target",
        ]
        .nunique()
    )


    unmatched_targets = (
        results_df.loc[
            results_df["candidate_type"] == "no_match",
            "ORPHAcode_target",
        ]
        .nunique()
    )


    selected_candidates = (
        results_df[
            results_df["is_selected"] == True
        ]
        .shape[0]
    )


    print(
        "\nUnannotated diseases with >=1 exact match: %d"
        % matched_targets
    )

    print(
        "Unannotated diseases with no exact match:  %d"
        % unmatched_targets
    )

    print(
        "Selected dominant/fully-supported HPO rows: %d"
        % selected_candidates
    )


print("\nDone.")